# Pengecekan Data `done_running` (10 skenario SRL-NER)

Notebook **validasi/sanity-check** untuk hasil run di `data/result/pseudo-labelling/SRL-NER/done_running/`. Bukan visualisasi error (itu di `visualize_error_predictions.ipynb`), tapi memastikan **datanya benar & konsisten** sebelum dipakai untuk Bab 4.

10 skenario: `augmentation, baseline, cahya-bert-base, distilbert, indobert-base-p1, jscl, pos-tag, roberta, scl, weighted-class`.

**Yang dicek:**
1. Kelengkapan file tiap skenario (correct/incorrect xlsx, iteration_log, runtime, models).
2. Total token = correct + incorrect → harus **42.558 token / 258 chunk** (test set sama untuk semua).
3. Validitas label (hanya `O` + B/I × {PERSON, LOCATION, EVENT, TIME}), cek NaN.
4. Akurasi token-level = correct / total, jumlah error.
5. Distribusi label gold **identik** antar-skenario (bukti test set tidak berubah).
6. Konvergensi self-training (iteration_log) + runtime.

Akhir: daftar peringatan otomatis kalau ada yang tidak beres.

## 0. Setup

In [ ]:
import re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
for _ in range(8):
    if (ROOT / 'CLAUDE.md').exists():
        break
    ROOT = ROOT.parent
DONE = ROOT / 'data/result/pseudo-labelling/SRL-NER/done_running'

EXPECT_TOKEN = 42558   # ukuran test set yang diharapkan
EXPECT_CHUNK = 258
VALID_TYPES = {'PERSON', 'LOCATION', 'EVENT', 'TIME'}

# nama skenario -> path INCORRECT (relatif done_running). correct = ganti 'incorrect'->'correct'.
SCN = {
    'augmentation':   'augmentation/output_S4_augmentation/evaluation/bert-only-sirah-ner-iterative-6-incorrect.xlsx',
    'baseline':       'baseline/output_S1_baseline/evaluation/bert-only-sirah-ner-iterative-6-incorrect.xlsx',
    'cahya-bert-base':'cahya-bert-base/output_GrupB_cahya/evaluation/bert-only-sirah-ner-iterative-6-incorrect.xlsx',
    'distilbert':     'distilbert/output_GrupB_distilbert/evaluation/bert-only-sirah-ner-iterative-6-incorrect.xlsx',
    'indobert-base-p1':'indobert-base-p1/output_GrupB_cased/evaluation/bert-only-sirah-ner-iterative-6-incorrect.xlsx',
    'jscl':           'jscl/output_S2b_jscl/evaluation/bert-only-sirah-ner-S2b-jscl-iterative-6-incorrect.xlsx',
    'pos-tag':        'pos-tag/output_pos_tag/evaluation/bert-pos-sirah-ner-iterative-4-incorrect.xlsx',
    'roberta':        'roberta/output_GrupB_roberta/evaluation/bert-only-sirah-ner-iterative-6-incorrect.xlsx',
    'scl':            'scl/output_S2a_scl/evaluation/bert-only-sirah-ner-S2a-scl-iterative-4-incorrect.xlsx',
    'weighted-class': 'weighted-class/drive-download-20260611T025134Z-3-001/output_S2_weighted_ce/evaluation/bert-only-sirah-ner-iterative-6-incorrect.xlsx',
}

WARN = []  # kumpulan peringatan, dicetak di akhir

# cek folder level-1 vs daftar SCN
folders = sorted([p.name for p in DONE.iterdir() if p.is_dir()])
print('Folder di done_running :', folders)
missing = [f for f in folders if f not in SCN]
if missing:
    WARN.append(f'Folder tidak terdaftar di SCN: {missing}')
    print('!! belum terdaftar:', missing)
else:
    print('OK: semua', len(folders), 'folder terdaftar di SCN.')

## 1. Kelengkapan file tiap skenario

In [ ]:
rows = []
for name, rel in SCN.items():
    inc = DONE / rel
    cor = Path(str(inc).replace('incorrect', 'correct'))
    ev = inc.parent
    itlog = ev / 'iteration_log.csv'
    rt = ev / 'runtime_skenario.txt'
    models = list((ev.parent / 'models').glob('*')) if (ev.parent / 'models').exists() else []
    rows.append({
        'skenario': name,
        'incorrect.xlsx': inc.exists(),
        'correct.xlsx': cor.exists(),
        'iteration_log': itlog.exists(),
        'runtime': rt.exists(),
        'n_model_ckpt': len(models),
    })
    for f, lbl in [(inc, 'incorrect'), (cor, 'correct')]:
        if not f.exists():
            WARN.append(f'{name}: {lbl}.xlsx TIDAK ADA ({f})')
files = pd.DataFrame(rows).set_index('skenario')
files

## 2-4. Total token, validitas label, akurasi token-level

In [ ]:
def etype(l):
    if not isinstance(l, str) or l == 'O':
        return 'O'
    return re.sub(r'^[BI][-_]', '', l)

COLS = {'Unnamed: 0', 'text_id', 'token', 'true_label', 'pred_label', 'text'}
gold_dist = {}   # simpan distribusi gold per skenario utk cek konsistensi
rows = []
for name, rel in SCN.items():
    inc = DONE / rel
    cor = Path(str(inc).replace('incorrect', 'correct'))
    di = pd.read_excel(inc); dc = pd.read_excel(cor) if cor.exists() else pd.DataFrame()
    d = pd.concat([di, dc], ignore_index=True)
    n_tot, n_inc = len(d), len(di)
    n_chunk = d['text_id'].nunique()
    # validitas kolom & label
    miss_cols = COLS - set(d.columns)
    types = set(d['true_label'].dropna().map(etype)) - {'O'}
    bad_types = types - VALID_TYPES
    n_nan = int(d['true_label'].isna().sum() + d['pred_label'].isna().sum())
    gold_dist[name] = d.assign(gt=d['true_label'].fillna('O').map(etype)) \
                       .query("gt != 'O'").groupby('gt').size().reindex(sorted(VALID_TYPES)).fillna(0).astype(int)
    rows.append({
        'skenario': name, 'total_token': n_tot, 'n_chunk': n_chunk,
        'n_correct': n_tot - n_inc, 'n_incorrect': n_inc,
        'token_acc_%': round((n_tot - n_inc) / n_tot * 100, 3) if n_tot else 0,
        'token_OK': n_tot == EXPECT_TOKEN, 'chunk_OK': n_chunk == EXPECT_CHUNK,
        'label_valid': not bad_types, 'kolom_lengkap': not miss_cols, 'n_NaN': n_nan,
    })
    if n_tot != EXPECT_TOKEN:
        WARN.append(f'{name}: total token {n_tot} != {EXPECT_TOKEN}')
    if n_chunk != EXPECT_CHUNK:
        WARN.append(f'{name}: n_chunk {n_chunk} != {EXPECT_CHUNK}')
    if bad_types:
        WARN.append(f'{name}: label tak dikenal {bad_types}')
    if miss_cols:
        WARN.append(f'{name}: kolom hilang {miss_cols}')

check = pd.DataFrame(rows).set_index('skenario').sort_values('token_acc_%', ascending=False)
check

## 5. Distribusi label gold identik antar-skenario?

Karena test set sama, jumlah token gold tiap kelas **harus sama** di semua skenario. Kalau beda → ada yang pakai test set berbeda.

In [ ]:
dist = pd.DataFrame(gold_dist).T
dist['TOTAL'] = dist.sum(axis=1)
display(dist)

nunique = dist.drop(columns='TOTAL').nunique()
if (nunique == 1).all():
    print('OK: distribusi gold IDENTIK di semua skenario (test set konsisten).')
else:
    beda = nunique[nunique > 1].index.tolist()
    WARN.append(f'Distribusi gold BERBEDA antar-skenario di kelas: {beda}')
    print('!! gold tidak identik di kelas:', beda)

## 6. Konvergensi self-training + runtime

In [ ]:
rows = []
for name, rel in SCN.items():
    ev = (DONE / rel).parent
    itlog = ev / 'iteration_log.csv'
    rt = ev / 'runtime_skenario.txt'
    rec = {'skenario': name, 'n_iter': None, 'last_n_above': None, 'konvergen': None, 'menit': None}
    if itlog.exists():
        lg = pd.read_csv(itlog)
        rec['n_iter'] = int(lg['iter'].max())
        rec['last_n_above'] = int(lg['n_above'].iloc[-1])
        rec['konvergen'] = bool((lg['n_above'] == 0).any())  # pernah mentok 0 = konvergen
    if rt.exists():
        m = re.search(r'menit=([0-9.]+)', rt.read_text())
        rec['menit'] = float(m.group(1)) if m else None
    rows.append(rec)
conv = pd.DataFrame(rows).set_index('skenario')
conv

## 7. Verdict / peringatan

In [ ]:
print(f'Skenario diperiksa : {len(SCN)}')
print(f'Total token target : {EXPECT_TOKEN} / {EXPECT_CHUNK} chunk\n')
if not WARN:
    print('SEMUA CEK LULUS — data done_running konsisten & lengkap.')
else:
    print(f'{len(WARN)} PERINGATAN:')
    for w in WARN:
        print('  -', w)